In [1]:
def main(datasources, start_date, end_date):
    """
    BigAlpha 2026：成交量承载方向因子
    Volume-Confirmed Conviction Factor (VCC)

    比较高成交量分钟与低成交量分钟的方向性价格冲击，并使用VWAP位置、
    价格路径效率、成交量集中度和流动性质量确认信号。

    返回列严格为：date, instrument, factor。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    sql = f"""
    WITH raw AS (
        SELECT
            date AS ts,
            date::DATE AS trade_date,
            strftime(date, '%Y-%m-%d') AS trading_day,
            instrument,
            close,
            volume,
            amount,
            ask_price1,
            bid_price1
        FROM {bar1m}
        WHERE close > 0
          AND volume >= 0
          AND amount >= 0
          AND ask_price1 > 0
          AND bid_price1 > 0
          AND ask_price1 >= bid_price1
          -- 09:31通常混合集合竞价成交，单独剔除以降低结构性噪声。
          AND date::TIME >= TIME '09:32:00'
          AND date::TIME <= TIME '15:00:00'
    ),

    lagged AS (
        SELECT
            *,
            LAG(close) OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
            ) AS prev_close,
            LAG(volume) OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
            ) AS prev_cum_volume,
            LAG(amount) OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
            ) AS prev_cum_amount
        FROM raw
    ),

    minute_bar AS (
        SELECT
            *,
            close / NULLIF(prev_close, 0) - 1.0 AS ret_1m,
            CASE
                WHEN prev_cum_volume IS NULL THEN GREATEST(volume, 0)
                WHEN volume >= prev_cum_volume THEN volume - prev_cum_volume
                ELSE GREATEST(volume, 0)
            END AS minute_volume,
            CASE
                WHEN prev_cum_amount IS NULL THEN GREATEST(amount, 0)
                WHEN amount >= prev_cum_amount THEN amount - prev_cum_amount
                ELSE GREATEST(amount, 0)
            END AS minute_amount,
            (ask_price1 - bid_price1)
                / NULLIF((ask_price1 + bid_price1) / 2.0, 0) AS rel_spread
        FROM lagged
    ),

    volume_thresholds AS (
        SELECT
            trading_day,
            instrument,
            quantile(minute_volume, 0.50) AS volume_q50,
            quantile(minute_volume, 0.80) AS volume_q80
        FROM minute_bar
        WHERE minute_volume IS NOT NULL
        GROUP BY trading_day, instrument
    ),

    tagged AS (
        SELECT
            m.*,
            v.volume_q50,
            v.volume_q80
        FROM minute_bar AS m
        JOIN volume_thresholds AS v
          ON m.trading_day = v.trading_day
         AND m.instrument = v.instrument
    ),

    daily_raw AS (
        SELECT
            trade_date,
            instrument,
            COUNT(*) AS bar_count,

            -- 高成交量分钟的方向一致性，范围约[-1, 1]。
            SUM(
                CASE WHEN minute_volume >= volume_q80
                     THEN ret_1m * SQRT(GREATEST(minute_volume, 0.0))
                     ELSE 0.0 END
            ) / NULLIF(
                SUM(
                    CASE WHEN minute_volume >= volume_q80
                         THEN ABS(ret_1m) * SQRT(GREATEST(minute_volume, 0.0))
                         ELSE 0.0 END
                ),
                0
            ) AS high_volume_direction,

            -- 低成交量分钟多为回撤时，将增强主方向的可信度。
            SUM(
                CASE WHEN minute_volume <= volume_q50
                     THEN ret_1m * SQRT(GREATEST(minute_volume, 0.0))
                     ELSE 0.0 END
            ) / NULLIF(
                SUM(
                    CASE WHEN minute_volume <= volume_q50
                         THEN ABS(ret_1m) * SQRT(GREATEST(minute_volume, 0.0))
                         ELSE 0.0 END
                ),
                0
            ) AS low_volume_direction,

            SUM(
                CASE WHEN minute_volume >= volume_q80
                     THEN minute_volume ELSE 0 END
            ) / NULLIF(SUM(minute_volume), 0) AS high_volume_share,

            FIRST(close ORDER BY ts) AS first_close,
            LAST(close ORDER BY ts) AS last_close,
            SUM(minute_amount) / NULLIF(SUM(minute_volume), 0) AS intraday_vwap,
            SUM(ABS(ret_1m)) AS path_length,
            AVG(rel_spread) AS avg_rel_spread
        FROM tagged
        GROUP BY trade_date, instrument
    ),

    quality AS (
        SELECT
            *,
            -- 强调高量推动、低量回撤的非对称结构。
            0.75 * COALESCE(high_volume_direction, 0.0)
                - 0.25 * COALESCE(low_volume_direction, 0.0)
                AS volume_impact_asymmetry,

            TANH(
                50.0 * (last_close / NULLIF(intraday_vwap, 0) - 1.0)
            ) AS vwap_confirmation,

            LEAST(
                1.0,
                GREATEST(
                    -1.0,
                    (last_close / NULLIF(first_close, 0) - 1.0)
                    / NULLIF(path_length, 0)
                )
            ) AS price_efficiency,

            LEAST(
                1.5,
                GREATEST(
                    0.5,
                    COALESCE(high_volume_share, 0.0) / 0.50
                )
            ) AS volume_concentration,

            1.0 / (1.0 + 200.0 * COALESCE(avg_rel_spread, 0.0))
                AS liquidity_quality
        FROM daily_raw
    ),

    combined AS (
        SELECT
            *,
            LEAST(
                1.0,
                GREATEST(
                    0.0,
                    0.5
                    + 0.25 * (
                        CASE WHEN volume_impact_asymmetry >= 0
                             THEN 1.0 ELSE -1.0 END
                    ) * COALESCE(vwap_confirmation, 0.0)
                    + 0.25 * (
                        CASE WHEN volume_impact_asymmetry >= 0
                             THEN 1.0 ELSE -1.0 END
                    ) * COALESCE(price_efficiency, 0.0)
                )
            ) AS direction_confirmation
        FROM quality
    )

    SELECT
        trade_date::DATETIME AS date,
        instrument,
        CASE
            WHEN bar_count >= 100 THEN TANH(
                2.5 * COALESCE(volume_impact_asymmetry, 0.0)
                * (0.5 + 0.5 * ABS(COALESCE(price_efficiency, 0.0)))
                * COALESCE(direction_confirmation, 0.5)
                * COALESCE(volume_concentration, 1.0)
                * COALESCE(liquidity_quality, 1.0)
            )
            ELSE 0.0
        END AS factor
    FROM combined
    ORDER BY trade_date, instrument
    """

    factor = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    factor["date"] = pd.to_datetime(factor["date"])
    factor = factor[["date", "instrument", "factor"]]
    factor["factor"] = pd.to_numeric(factor["factor"], errors="coerce")
    factor["factor"] = (
        factor["factor"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    # 以官方股票池为左表，停牌或数据不足的成分股赋0。
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])
    factor = pd.merge(
        stk_pool[["date", "instrument"]],
        factor,
        how="left",
        on=["date", "instrument"],
    )
    factor["factor"] = factor["factor"].fillna(0.0)
    factor = (
        factor[["date", "instrument", "factor"]]
        .drop_duplicates(["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )

    if list(factor.columns) != ["date", "instrument", "factor"]:
        raise ValueError("输出列必须且只能是 date、instrument、factor")
    if factor.isna().any().any():
        raise ValueError("因子输出中不应存在缺失值")

    return factor




if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-08-04 11:01:11] [info     ] 计算因子，区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
